In [ ]:
import pandas as pd
import requests
import sqlite3
from functools import reduce
from pathlib import Path

# =========================
# 1. CONFIGURAÇÃO
# =========================

CAMINHO_DB = Path("../database/renner_credit_analysis.db")

DATA_INICIAL = "01/01/2021"
DATA_FINAL = "31/12/2025"

series_bcb = {
    "selic_meta": 432,
    "ipca": 433,
    "inadimplencia_pf_total": 21084,
    "comprometimento_renda_familias": 29034,
    "endividamento_familias": 29037
}

# =========================
# 2. FUNÇÃO PARA BAIXAR SGS
# =========================

def baixar_serie_bcb(nome_serie, codigo, data_inicial, data_final):
    url = (
        f"https://api.bcb.gov.br/dados/serie/bcdata.sgs.{codigo}/dados"
        f"?formato=json&dataInicial={data_inicial}&dataFinal={data_final}"
    )

    resposta = requests.get(url, timeout=30)
    resposta.raise_for_status()

    dados = resposta.json()

    df = pd.DataFrame(dados)

    if df.empty:
        print(f"Atenção: série vazia - {nome_serie} ({codigo})")
        return pd.DataFrame(columns=["data", "valor", "serie", "codigo_sgs"])

    df["data"] = pd.to_datetime(df["data"], format="%d/%m/%Y")
    df["valor"] = df["valor"].str.replace(",", ".", regex=False).astype(float)
    df["serie"] = nome_serie
    df["codigo_sgs"] = codigo

    return df[["data", "valor", "serie", "codigo_sgs"]]

# =========================
# 3. BAIXAR TODAS AS SÉRIES
# =========================

lista_dfs = []

for nome, codigo in series_bcb.items():
    print(f"Baixando {nome} - SGS {codigo}")
    df_temp = baixar_serie_bcb(nome, codigo, DATA_INICIAL, DATA_FINAL)
    lista_dfs.append(df_temp)

raw_macro_bcb = pd.concat(lista_dfs, ignore_index=True)

# =========================
# 4. CRIAR PERÍODO TRIMESTRAL
# =========================

raw_macro_bcb["periodo"] = raw_macro_bcb["data"].dt.to_period("Q").dt.to_timestamp("Q")

# =========================
# 5. FUNÇÃO DE AGREGAÇÃO TRIMESTRAL
# =========================

def agregar_serie_trimestre(df, nome_serie, tipo_agregacao):
    temp = df[df["serie"] == nome_serie].copy()

    if tipo_agregacao == "media":
        saida = (
            temp.groupby("periodo", as_index=False)["valor"]
            .mean()
            .rename(columns={"valor": nome_serie})
        )

    elif tipo_agregacao == "soma":
        saida = (
            temp.groupby("periodo", as_index=False)["valor"]
            .sum()
            .rename(columns={"valor": nome_serie})
        )

    elif tipo_agregacao == "ultimo":
        temp = temp.sort_values("data")
        saida = (
            temp.groupby("periodo", as_index=False)
            .tail(1)[["periodo", "valor"]]
            .rename(columns={"valor": nome_serie})
        )

    else:
        raise ValueError("tipo_agregacao deve ser: media, soma ou ultimo")

    return saida

# =========================
# 6. AGREGAR CADA VARIÁVEL
# =========================

selic_tri = agregar_serie_trimestre(raw_macro_bcb, "selic_meta", "media")
ipca_tri = agregar_serie_trimestre(raw_macro_bcb, "ipca", "soma")

inadimplencia_tri = agregar_serie_trimestre(raw_macro_bcb, "inadimplencia_pf_total", "ultimo")
comprometimento_tri = agregar_serie_trimestre(raw_macro_bcb, "comprometimento_renda_familias", "ultimo")
endividamento_tri = agregar_serie_trimestre(raw_macro_bcb, "endividamento_familias", "ultimo")

dfs_trimestrais = [
    selic_tri,
    ipca_tri,
    inadimplencia_tri,
    comprometimento_tri,
    endividamento_tri
]

stg_macro_quarterly = reduce(
    lambda left, right: pd.merge(left, right, on="periodo", how="outer"),
    dfs_trimestrais
)

stg_macro_quarterly = stg_macro_quarterly.sort_values("periodo")

# =========================
# 7. CRIAR INDICADORES 12M
# =========================

stg_macro_quarterly["ipca_acum_12m"] = (
    stg_macro_quarterly["ipca"]
    .rolling(window=4)
    .sum()
)

for coluna in [
    "selic_meta",
    "inadimplencia_pf_total",
    "comprometimento_renda_familias",
    "endividamento_familias"
]:
    stg_macro_quarterly[f"{coluna}_lag_12m"] = stg_macro_quarterly[coluna].shift(4)
    stg_macro_quarterly[f"{coluna}_delta_12m"] = (
        stg_macro_quarterly[coluna] - stg_macro_quarterly[f"{coluna}_lag_12m"]
    )

# =========================
# 8. AJUSTAR FORMATO DO PERÍODO
# =========================

raw_macro_bcb["data"] = raw_macro_bcb["data"].dt.strftime("%Y-%m-%d")
raw_macro_bcb["periodo"] = raw_macro_bcb["periodo"].dt.strftime("%Y-%m-%d")

stg_macro_quarterly["periodo"] = stg_macro_quarterly["periodo"].dt.strftime("%Y-%m-%d")

# =========================
# 9. SALVAR NO SQLITE
# =========================

conn = sqlite3.connect(CAMINHO_DB)

raw_macro_bcb.to_sql("raw_macro_bcb", conn, if_exists="replace", index=False)
stg_macro_quarterly.to_sql("stg_macro_quarterly", conn, if_exists="replace", index=False)

conn.close()

print("Processo finalizado.")
print("Tabelas criadas:")
print("- raw_macro_bcb")
print("- stg_macro_quarterly")

Baixando selic_meta - SGS 432


Baixando ipca - SGS 433
Baixando inadimplencia_pf_total - SGS 21084
Baixando comprometimento_renda_familias - SGS 29034
Baixando endividamento_familias - SGS 29037
Processo finalizado.
Tabelas criadas:
- raw_macro_bcb
- stg_macro_quarterly
